## Model Pipeline Test
### Baseline Model vs Resnet

Implementation of confusion matrix and focus on recall to not receive false negatives

In [ ]:
# External Imports
import sys
from pathlib import Path
import glob
import torch
import torch.optim as optim
import torch.nn as nn
from torchinfo import summary
from sklearn import metrics
import seaborn as sns

# Internal Imports
sys.path.insert(0, '../src')
from src.Dataset.mri_split import split_patients
from src.Dataset.data_loaders import get_dataloaders
from src.Dataset.cache import load_cache
from src.Model.baseline_cnn import BaselineModel
from src.Model.mri_resnet import build_resnet
from src.Model.persistance import load_weights
from src.Model.evaluation import evaluate

In [ ]:
accepted_data = {}
rejected_data = {}
try:
    accepted_data = load_cache(Path("../data/processed/cache/accepted_data.json"), Path.cwd().parent)
    rejected_data = load_cache(Path("../data/processed/cache/rejected_data.json"), Path.cwd().parent)
except BaseException as e:
    print(e)

In [ ]:
#print(accepted_data.keys())
#print(accepted_data['TCGA_CS_4941_19960909'])

In [ ]:
SPLIT_SEED = 42
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

In [ ]:
BATCH_SIZE = 32
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(train_patients, val_patients, test_patients, batch_size=BATCH_SIZE)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
resnet_model = build_resnet(num_classes=2)
load_weights(resnet_model, Path("../models/resnet18_2026-03-21_11h03-39s.pth"))
resnet_model.to(device)

In [ ]:
predictions, labels = evaluate(resnet_model, test_dataloader, device)

In [ ]:
predictions

In [ ]:
labels

In [ ]:
confusion_matrix = metrics.confusion_matrix(labels, predictions)
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=["No Tumor", "Tumor"], yticklabels=["No Tumor", "Tumor"])

In [ ]:
results = metrics.classification_report(labels, predictions, target_names=["No Tumor", "Tumor"])
print(results)